<a href="https://colab.research.google.com/github/FahimSS999/IoT-Based-Smart-Irrigation-System-with-Environmental-Sensing-and-Monitoring/blob/main/AI_Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random
import heapq
from collections import deque

# ==================== Utility ====================
def clear():
    print("\n" + "-"*50 + "\n")

# ==================== Trial 1: A* Pathfinding ====================
def astar(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal)}

    while open_set:
        _, current = heapq.heappop(open_set)
        if current == goal:
            return reconstruct_path(came_from, current)

        for neighbor in neighbors(current, rows, cols):
            if grid[neighbor[0]][neighbor[1]] == 1:
                continue
            tentative_g = g_score[current] + 1
            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score[neighbor], neighbor))
    return []

def heuristic(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def neighbors(pos, rows, cols):
    r, c = pos
    for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < rows and 0 <= nc < cols:
            yield (nr, nc)

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

def trial_astar():
    clear()
    print("Trial 1: Escape the Maze (A*)")
    grid = [
        [0,0,0,0,1],
        [1,1,0,1,0],
        [0,0,0,0,0],
        [0,1,1,1,0],
        [0,0,0,1,0]
    ]
    start, goal = (0,0), (4,4)
    path = astar(grid, start, goal)
    if path:
        print("AI found path:", path)
    else:
        print("No path found!")
    input("Press Enter for next trial...")

# ==================== Trial 2: 8 Puzzle (A*) ====================
goal_state = [[1,2,3],[4,5,6],[7,8,0]]

def flatten(state):
    return tuple(num for row in state for num in row)

def manhattan(state):
    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                gr, gc = divmod(val-1, 3)
                distance += abs(r-gr) + abs(c-gc)
    return distance

def get_moves(state):
    moves = []
    r, c = [(r,c) for r in range(3) for c in range(3) if state[r][c]==0][0]
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = [list(row) for row in state]
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            moves.append(new_state)
    return moves

def astar_8puzzle(start):
    open_set = []
    heapq.heappush(open_set, (manhattan(start), 0, start, []))
    visited = set()
    while open_set:
        _, g, state, path = heapq.heappop(open_set)
        if state == goal_state:
            return path
        visited.add(flatten(state))
        for move in get_moves(state):
            if flatten(move) not in visited:
                heapq.heappush(open_set, (g+1+manhattan(move), g+1, move, path+[move]))
    return []

def trial_8puzzle():
    clear()
    print("Trial 2: The Sliding Relic Puzzle (8 Puzzle with A*)")
    start_state = [[1,2,3],[4,0,6],[7,5,8]]
    solution = astar_8puzzle(start_state)
    print("AI solved the puzzle in", len(solution), "moves.")
    input("Press Enter for next trial...")

# ==================== Trial 3: Genetic Algorithm ====================
def trial_genetic():
    clear()
    print("Trial 3: The Evolving Beast (Genetic Algorithm)")
    print("Rock-Paper-Scissors. First to 3 wins. AI adapts to your moves!")
    history = []
    ai_pref = {"rock":1, "paper":1, "scissors":1}

    def ai_move():
        if not history:
            return random.choice(["rock","paper","scissors"])
        # predict player's next move as most frequent so far
        player_guess = max(ai_pref, key=ai_pref.get)
        counter = {"rock":"paper", "paper":"scissors", "scissors":"rock"}
        return counter[player_guess]

    player_score, ai_score = 0, 0
    while player_score < 3 and ai_score < 3:
        player = input("Your move (rock/paper/scissors): ").strip().lower()
        if player not in ["rock","paper","scissors"]:
            continue
        ai = ai_move()
        print("AI chose:", ai)
        if player == ai:
            print("Draw!")
        elif (player=="rock" and ai=="scissors") or (player=="paper" and ai=="rock") or (player=="scissors" and ai=="paper"):
            print("You win this round!")
            player_score += 1
        else:
            print("AI wins this round!")
            ai_score += 1
        history.append(player)
        ai_pref[player] += 1
    if player_score > ai_score:
        print("You defeated the evolving beast!")
    else:
        print("The beast outsmarted you!")
    input("Press Enter for next trial...")

# ==================== Trial 4: Minimax ====================
def minimax_ttt(board, depth, is_max):
    scores = {"X":1, "O":-1, "draw":0}
    result = check_winner(board)
    if result:
        return scores[result]
    if is_max:
        best = -float('inf')
        for i in range(9):
            if board[i] == " ":
                board[i] = "X"
                best = max(best, minimax_ttt(board, depth+1, False))
                board[i] = " "
        return best
    else:
        best = float('inf')
        for i in range(9):
            if board[i] == " ":
                board[i] = "O"
                best = min(best, minimax_ttt(board, depth+1, True))
                board[i] = " "
        return best

def best_move(board):
    best_val = -float('inf')
    move = -1
    for i in range(9):
        if board[i] == " ":
            board[i] = "X"
            move_val = minimax_ttt(board, 0, False)
            board[i] = " "
            if move_val > best_val:
                best_val = move_val
                move = i
    return move

def check_winner(board):
    wins = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for a,b,c in wins:
        if board[a] == board[b] == board[c] and board[a] != " ":
            return board[a]
    if " " not in board:
        return "draw"
    return None

def print_board(board):
    print(f"{board[0]}|{board[1]}|{board[2]}")
    print("-+-+-")
    print(f"{board[3]}|{board[4]}|{board[5]}")
    print("-+-+-")
    print(f"{board[6]}|{board[7]}|{board[8]}")

def trial_minimax():
    clear()
    print("Trial 4: The Oracle's Game (Tic-Tac-Toe with Minimax)")
    board = [" "]*9
    while True:
        print_board(board)
        move = int(input("Your move (0-8): "))
        if board[move] != " ":
            continue
        board[move] = "O"
        if check_winner(board):
            break
        ai_move = best_move(board)
        board[ai_move] = "X"
        if check_winner(board):
            break
    print_board(board)
    result = check_winner(board)
    if result == "O":
        print("You win!")
    elif result == "X":
        print("AI wins!")
    else:
        print("It's a draw!")
    input("Press Enter for next trial...")

# ==================== Trial 5: Alpha-Beta Pruning ====================
def alpha_beta_nim(n, alpha, beta, maximizing):
    if n == 0:
        return -1 if maximizing else 1
    if maximizing:
        value = -float('inf')
        for m in range(1,4):
            if n-m >= 0:
                value = max(value, alpha_beta_nim(n-m, alpha, beta, False))
                alpha = max(alpha, value)
                if alpha >= beta:
                    break
        return value
    else:
        value = float('inf')
        for m in range(1,4):
            if n-m >= 0:
                value = min(value, alpha_beta_nim(n-m, alpha, beta, True))
                beta = min(beta, value)
                if beta <= alpha:
                    break
        return value

def trial_alphabeta():
    clear()
    print("Trial 5: The Matchstick Game (Alpha-Beta Pruning)")
    n = 21
    while n > 0:
        print(f"Sticks left: {n}")
        take = int(input("Take 1-3 sticks: "))
        if take not in [1,2,3] or take > n:
            continue
        n -= take
        if n == 0:
            print("You took the last stick. You lose!")
            return
        # AI turn
        for m in range(1,4):
            if n-m >= 0 and alpha_beta_nim(n-m, -float('inf'), float('inf'), False) == 1:
                ai_take = m
                break
        else:
            ai_take = 1
        print(f"AI takes {ai_take} sticks.")
        n -= ai_take
        if n == 0:
            print("AI took the last stick. AI loses! You win!")
            return

# ==================== Main ====================
def main():
    print("Welcome to AI Quest: The Five Trials!")
    input("Press Enter to begin...")
    trial_astar()
    trial_8puzzle()
    trial_genetic()
    trial_minimax()
    trial_alphabeta()
    print("\nCongratulations! You have completed all five trials!")

main()


Welcome to AI Quest: The Five Trials!
Press Enter to begin...

--------------------------------------------------

Trial 1: Escape the Maze (A*)
AI found path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
Press Enter for next trial...

--------------------------------------------------

Trial 2: The Sliding Relic Puzzle (8 Puzzle with A*)
AI solved the puzzle in 2 moves.
Press Enter for next trial...

--------------------------------------------------

Trial 3: The Evolving Beast (Genetic Algorithm)
Rock-Paper-Scissors. First to 3 wins. AI adapts to your moves!
Your move (rock/paper/scissors): rock
AI chose: paper
AI wins this round!
Your move (rock/paper/scissors): paper
AI chose: paper
Draw!
Your move (rock/paper/scissors): scissors
AI chose: paper
You win this round!
Your move (rock/paper/scissors): scissors
AI chose: paper
You win this round!
Your move (rock/paper/scissors): scissors
AI chose: rock
AI wins this round!
Your move (rock/paper/scissors): sci